# Sri Sivasubramaniya Nadar College of Engineering, Chennai
### (An Autonomous Institution Affiliated to Anna University)
**Degree & Branch:** M. Tech (Integrated) Computer Science & Engineering | **Semester:** V  
**Subject Code & Name:** ICS1512 & Machine Learning Algorithms Laboratory  
**Academic Year:** 2026-2027 (Odd) | **Batch:** 2024-2029  

---
## Experiment 7: Dimensionality Reduction and Model Evaluation (With and Without PCA)

**Student Name:** Danusu K | **Register Number:** 3122247001013  
**Faculty:** Dr. Poreddy Ajay Kumar Reddy  

### Objectives:
1. Study the mathematical formulation and effect of dimensionality reduction using **Principal Component Analysis (PCA)**.
2. Train, optimize, and validate **10 Machine Learning Classifiers** under two distinct paradigms:
   - **No-PCA:** Original 30 continuous standardized features.
   - **With-PCA:** Reduced orthogonal feature space (10 Principal Components capturing >= 95% variance).
3. Perform systematic hyperparameter tuning using **5-Fold Stratified Cross-Validation**.
4. Assess model accuracy, stability (cross-fold standard deviation), execution latency, and test generalization on the **Wisconsin Diagnostic Breast Cancer (WDBC)** dataset.
5. Deconstruct when PCA aids classification performance and when it introduces loss of non-linear discrimination.


## 1. Environment Setup & Library Imports
Import essential statistical, machine learning, and visualization libraries.

In [1]:
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, log_loss
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment initialized successfully with seed = 42.")


## 2. Dataset Ingestion & Preprocessing
Load the Wisconsin Diagnostic Breast Cancer (WDBC) dataset, standardize continuous attributes, and partition into stratified 80% train and 20% test splits.

In [2]:
feature_names = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
    "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
    "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

df = pd.read_csv("../dataset/wdbc.csv")
if "id" in df.columns:
    df = df.drop(columns=["id"])
if "Unnamed: 32" in df.columns:
    df = df.drop(columns=["Unnamed: 32"])

X = df[feature_names].copy()
y = df["diagnosis"].map({"M": 1, "B": 0}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}")
print(f"Train class balance: Benign = {(1-y_train).sum()}, Malignant = {y_train.sum()}")


## 3. Principal Component Analysis (PCA) Decomposition
Compute the covariance matrix and solve the characteristic eigenvalue equation. Evaluate the Scree plot, Kaiser criterion, and select the 95% variance target.

In [3]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train_scaled)

var_exp = pca_full.explained_variance_ratio_ * 100
cum_var_exp = np.cumsum(var_exp)
eigenvalues = pca_full.explained_variance_

n_selected = int(np.argmax(cum_var_exp >= 95.0) + 1)
pca_10 = PCA(n_components=n_selected, random_state=RANDOM_STATE)
X_train_pca = pca_10.fit_transform(X_train_scaled)
X_test_pca = pca_10.transform(X_test_scaled)

print(f"Chosen components: {n_selected} | Cumulative Variance Explained: {cum_var_exp[n_selected-1]:.2f}%")
print(f"Dimensionality Reduction: {(1 - n_selected / 30)*100:.1f}% reduction (30D -> 10D)")


## 4. Hyperparameter Tuning & 5-Fold Stratified Cross-Validation Results
Load and inspect the complete cross-validation results across all 10 models.

In [4]:
with open("experiment7_results.json", "r") as f:
    results = json.load(f)

cv_df = []
for m, v in results["cv_5fold_results"].items():
    folds_no = [f"{x*100:.1f}%" for x in v["no_pca"]["folds"]]
    cv_df.append({
        "Model": m.replace("_", " "),
        "Fold 1": folds_no[0],
        "Fold 2": folds_no[1],
        "Fold 3": folds_no[2],
        "Fold 4": folds_no[3],
        "Fold 5": folds_no[4],
        "Avg (No-PCA)": f"{v['no_pca']['mean_accuracy']*100:.2f}% ± {v['no_pca']['std_accuracy']*100:.2f}%",
        "Avg (With-PCA)": f"{v['with_pca']['mean_accuracy']*100:.2f}% ± {v['with_pca']['std_accuracy']*100:.2f}%",
        "Delta Acc": f"{v['accuracy_delta_pct']:+.2f}%",
        "Stability Gain (Δσ)": f"{v['std_reduction']*100:+.2f}%"
    })

df_cv = pd.DataFrame(cv_df)
df_cv


In [5]:
test_df = []
for m, v in results["test_results"].items():
    test_df.append({
        "Model": m.replace("_", " "),
        "Test Acc (No-PCA)": f"{v['no_pca']['accuracy']*100:.2f}%",
        "Test Acc (With-PCA)": f"{v['with_pca']['accuracy']*100:.2f}%",
        "F1 (No-PCA)": f"{v['no_pca']['f1_score']*100:.2f}%",
        "F1 (With-PCA)": f"{v['with_pca']['f1_score']*100:.2f}%",
        "ROC-AUC (With-PCA)": f"{v['with_pca']['roc_auc']:.4f}",
        "Train Time (No-PCA)": f"{v['no_pca']['train_time_sec']*1000:.1f} ms",
        "Train Time (With-PCA)": f"{v['with_pca']['train_time_sec']*1000:.1f} ms"
    })

df_test = pd.DataFrame(test_df)
df_test


## 5. Visualizations & Diagnostic Comparisons
Examine the generated publication plots.

In [6]:
from IPython.display import Image, display
plot_files = [
    "../output_plots/01_scree_and_cumulative_variance.png",
    "../output_plots/02_pca_2d_and_3d_projections.png",
    "../output_plots/06_5fold_cv_foldwise_comparison.png",
    "../output_plots/07_model_performance_comparison_bars.png",
    "../output_plots/08_training_time_and_speedup.png",
    "../output_plots/09_confusion_matrices.png",
    "../output_plots/10_roc_curves.png"
]
for p in plot_files:
    if os.path.exists(p):
        print(f"Displaying: {os.path.basename(p)}")
        display(Image(filename=p, width=800))


## 6. Key Conclusions & Insights
1. **Multicollinearity Elimination:** Dimensionality reduction with PCA (=10$, .21\%$ variance) completely eliminated severe correlation among nuclear attributes, improving linear model conditioning and boosting **Logistic Regression** and **Decision Tree** cross-validation accuracy.
2. **Stability Enhancement:** Cross-fold standard deviation decreased for the majority of models (e.g. AdaBoost $\sigma: 1.12\% 	o 0.54\%$, Decision Tree $\sigma: 2.45\% 	o 1.64\%$, SVM $\sigma: 1.20\% 	o 0.82\%$), demonstrating variance reduction across fold partitions.
3. **Computational Efficiency:** Tree-based ensemble training latency was reduced by **.16	imes - 2.42	imes* with negligible impact on final classification discrimination.
4. **Stacking Robustness:** The Stacking ensemble achieved the highest overall 5-fold CV score (**98.02%**) under both No-PCA and With-PCA configurations, proving that multi-model aggregation is highly robust to subspace projections.
